# ラベル付きデータなしで分類する（ゼロショット分類、NLI）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/11_classification/zero_shot_nli.ipynb)

NLP クックブック 11-13 ／ テキスト分類

学習データがまったくない状態で、自分で決めたカテゴリにテキストを分類したい。

In [1]:
from transformers import PreTrainedTokenizerFast, pipeline

name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
# transformers 5 系の DebertaV2Tokenizer は全角の記号や英数字を [UNK] にしてしまう (issue #49622) ので、tokenizer.json から直接読む
tokenizer = PreTrainedTokenizerFast.from_pretrained(name)
clf = pipeline("zero-shot-classification", model=name, tokenizer=tokenizer)

labels = ["スポーツ", "経済", "テクノロジー", "料理", "政治"]
texts = ["大谷翔平が今季 50 本目のホームランを放った。",
         "トヨタ自動車の純利益は前年比 20% 増加した。",
         "新しい言語モデルは従来より少ない計算量で高い性能を達成した。",
         "玉ねぎを飴色になるまで炒めてから、カレールーを加えます。"]
for text in texts:
    r = clf(text, labels, hypothesis_template="この文は{}に関するものです。")
    print(f"{r['labels'][0]}({r['scores'][0]:.2f}) / {r['labels'][1]}({r['scores'][1]:.2f})  {text}")

スポーツ(0.65) / 政治(0.10)  大谷翔平が今季 50 本目のホームランを放った。
経済(0.91) / 料理(0.03)  トヨタ自動車の純利益は前年比 20% 増加した。
テクノロジー(0.38) / 経済(0.22)  新しい言語モデルは従来より少ない計算量で高い性能を達成した。
料理(0.99) / テクノロジー(0.00)  玉ねぎを飴色になるまで炒めてから、カレールーを加えます。


In [2]:
# 複数ラベルが同時に成り立つ場合 (各ラベルを独立に判定)
r = clf("新型スマートフォンの販売が好調で、アップルの株価が上昇した。", labels,
        hypothesis_template="この文は{}に関するものです。", multi_label=True)
print([(l, round(s, 2)) for l, s in zip(r["labels"], r["scores"])])

[('テクノロジー', 1.0), ('経済', 1.0), ('政治', 0.0), ('スポーツ', 0.0), ('料理', 0.0)]


- 仕組み: 「この文は◯◯に関するものです」という仮説が入力文から含意されるかを NLI モデルに判定させ、含意の確率が最も高いラベルを選ぶ。
- `hypothesis_template` は対象の言語で書く。言い回しによって精度が変わるので、いくつか試す。
- ラベル数だけモデルを実行するので遅い。大量に処理するなら、次項の埋め込みによる方法か、ゼロショットの結果を教師にして小さなモデルを学習する。